# Week 2 Day 2 — LangChain: Tools, Chains, Memory & Your First Framework Agent

**Goal:** Rebuild yesterday's agent with LangChain — chaining prompts, real memory, multiple tools (including external data), structured output, and error handling.

Uses the **live Gemini API** (`GEMINI_API_KEY` in `.env` or `../day 1/.env`). No mock.

| Day 1 (raw Python) | LangChain equivalent |
|--------------------|----------------------|
| `genai.Client` | `ChatGoogleGenerativeAI` |
| `TOOL_SCHEMAS` + `TOOL_FNS` | `@tool` / `StructuredTool` |
| `while` loop in `agent.py` | `AgentExecutor` |
| manual `messages` list | `ConversationBufferMemory` |


In [ ]:
%pip install -q -r requirements.txt


In [15]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
from dotenv import load_dotenv

HERE = Path.cwd() if (Path.cwd() / "tools.py").is_file() else Path.cwd() / "week 2" / "day 2"
load_dotenv(HERE / ".env")
load_dotenv(HERE.parent / "day 1" / ".env", override=True)
sys.path.insert(0, str(HERE))

from config import user_api_key, default_model, build_llm

key = user_api_key()
if not key:
    raise RuntimeError(f"Missing GEMINI_API_KEY in {HERE / '.env'} or ../day 1/.env")

print(f"Environment ready | Model: {default_model()} | Key: ...{key[-4:]}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Environment ready | Model: gemini-3.5-flash-lite | Key: ...p2dA


## Task 1 — LangChain setup & core concepts

LangChain wraps the LLM as a **Runnable**, tools as **schema-bearing callables**, and the agent loop as **AgentExecutor**.

**LCEL** (LangChain Expression Language) composes runnables with `|`. Under the hood, `prompt | llm | parser` builds a single `RunnableSequence`: each step's output becomes the next step's input, with streaming/callbacks/batch support inherited from `langchain-core` — you declare *data flow*, not manual glue code.


In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = build_llm()
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant."),
    ("human", "{question}"),
])

# LCEL pipeline: template -> model -> string parser
chain = prompt | llm | StrOutputParser()
answer = chain.invoke({"question": "In one sentence, what does LCEL's pipe operator compose?"})
print(answer)


c:\Users\pc 1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


The LCEL pipe operator (`|`) composes Runnables into a seamless sequence of execution, passing the output of one component as the input to the next.


In [17]:
# Concept map printed for the write-up
concept_map = {
    "LLM wrapper": "ChatGoogleGenerativeAI (Runnable)",
    "Tool": "@tool decorator — docstring becomes model-facing schema text",
    "AgentExecutor": "ReAct loop: plan -> tool.run -> append observation -> repeat",
    "Memory": "ConversationBufferMemory or RunnableWithMessageHistory",
}
for k, v in concept_map.items():
    print(f"{k:16} -> {v}")


LLM wrapper      -> ChatGoogleGenerativeAI (Runnable)
Tool             -> @tool decorator — docstring becomes model-facing schema text
AgentExecutor    -> ReAct loop: plan -> tool.run -> append observation -> repeat
Memory           -> ConversationBufferMemory or RunnableWithMessageHistory


## Task 2 — Define & register tools

Tool **docstrings** are not just documentation for humans — LangChain copies them into the JSON schema the model sees when choosing tools. Clear docstrings = better tool selection.

- **Reused from Day 1:** `calculator`, `get_weather`
- **New data source:** `lookup_product_price` reads `data/products.json`


In [18]:
from tools import calculator, get_weather, lookup_product_price, CORE_TOOLS

print("Tool names:", [t.name for t in CORE_TOOLS])
print("\n--- calculator schema excerpt ---")
print(calculator.description[:120], "...")
print("\n--- lookup_product_price schema excerpt ---")
print(lookup_product_price.description)

# Smoke test the JSON catalog tool (no LLM)
print("\nLive lookup:", lookup_product_price.invoke({"product_name": "Pro"}))


Tool names: ['calculator', 'get_weather', 'lookup_product_price']

--- calculator schema excerpt ---
Evaluate a basic arithmetic expression and return the numeric result as text.

Use for comparisons that need subtraction ...

--- lookup_product_price schema excerpt ---
Look up monthly USD price for a SaaS product from the local JSON catalog.

Reads data/products.json (no network). Product names are case-insensitive.
Returns price, tier, and a one-line description when found.

Live lookup: Pro: $49/month (standard) — Up to 10 users, 100 GB storage, priority support.


## Task 3 — Agent with create_tool_calling_agent + AgentExecutor

`verbose=True` prints the chain. Annotate mentally:
- **Reason** — model reads prompt + scratchpad
- **Act** — `Invoking: tool_name(...)`
- **Observe** — tool return value printed before the next model call


In [19]:
from agent_setup import build_agent_executor, message_text
from tools import CORE_TOOLS

question = (
    "Look up the weather in Karachi and London. "
    "Which city is warmer, and by how many degrees Celsius?"
)

executor = build_agent_executor(CORE_TOOLS, verbose=True, max_iterations=8)
result = executor.invoke({"input": question, "chat_history": []})
final = message_text(result["output"])
print("\n=== FINAL ===\n", final)




> Entering new AgentExecutor chain...


c:\Users\pc 1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_weather` with `{'city': 'Karachi'}`


Karachi: 33 C, hot, humid
Invoking: `get_weather` with `{'city': 'London'}`


London: 12 C, overcast, light rain

c:\Users\pc 1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `calculator` with `{'expression': '33 - 12'}`


21

c:\Users\pc 1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Karachi is warmer than London. Karachi has a temperature of 33°C, while London is at 12°C, making Karachi 21 degrees Celsius warmer than London.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPZUJFUF0IssG1vlbjdGcRy9DjUxCMLzECSysq8SNS//+2+XEyknHBm23Qlppyj+CdXRbh2lA6ImqNQb+woKqoXL+/KULIsqjhSUFfW5xQ+rzrNSo+hly6'}}]

> Finished chain.

=== FINAL ===
 Karachi is warmer than London. Karachi has a temperature of 33°C, while London is at 12°C, making Karachi 21 degrees Celsius warmer than London.


In [20]:
# Compare to Day 1: similar ReAct shape, but LangChain hides message formatting.
comparison = '''
Day 1 raw log tags:     [reason] [act] [observe] [final]
LangChain verbose:      > Entering AgentExecutor chain / Invoking: `tool` / observation text

Similar: model picks tools, multi-step until done.
Hidden:  scratchpad assembly, Gemini thought signatures, HTTP payloads.
'''
print(comparison)



Day 1 raw log tags:     [reason] [act] [observe] [final]
LangChain verbose:      > Entering AgentExecutor chain / Invoking: `tool` / observation text

Similar: model picks tools, multi-step until done.
Hidden:  scratchpad assembly, Gemini thought signatures, HTTP payloads.



## Task 4 — Conversation memory (3-turn follow-up)

Scenario:
1. "Find the price of Pro"
2. "Now compare it to Enterprise"
3. "Which one should I recommend to a budget-conscious client?"


In [21]:
from agent_setup import build_memory_executor, message_text

mem_executor = build_memory_executor(CORE_TOOLS, verbose=True)

turns = [
    "Find the monthly price of the Pro plan.",
    "Now compare it to Enterprise.",
    "Which one should I recommend to a budget-conscious client?",
]

last = None

for i, user_msg in enumerate(turns, 1):
    print(f"\n{'='*60}\nTURN {i}: {user_msg}\n{'='*60}")
    
    out = mem_executor.invoke({"input": user_msg})
    
    print("\nAssistant:", message_text(out["output"]))
    
    last = out

d:\internship\week 2\day 2\agent_setup.py:76: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
c:\Users\pc 1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



TURN 1: Find the monthly price of the Pro plan.


> Entering new AgentExecutor chain...

Invoking: `lookup_product_price` with `{'product_name': 'Pro plan'}`


ERROR: unknown product 'Pro plan'. Known: enterprise, pro, starter

c:\Users\pc 1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `lookup_product_price` with `{'product_name': 'pro'}`


Pro: $49/month (standard) — Up to 10 users, 100 GB storage, priority support.

c:\Users\pc 1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


...


Invoking: `calculator` with `{'expression': '149 - 49'}`


100

c:\Users\pc 1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The Pro plan costs **$49/month** (standard tier), while the Enterprise plan costs **$149/month** (premium tier). \n\nComparing the two:\n* **Price Difference:** The Enterprise plan is **$100 more per month** than the Pro plan.\n* **Pro Plan Features:** Up to 10 users, 100 GB storage, and priority support.\n* **Enterprise Plan Features:** Unlimited users, SSO, and a dedicated success manager.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPfNL5Syq4IfWBQKXeSOx4b4LeGK8BpWSG7KeEFPxeelNGNEBmhkuwYtTjNeC+eExnQ0AUFKj2fmahv/smaqvXLSxxICCeDbZgL20gOH6Eb69iHDSOYIUs'}}]

> Finished chain.

Assistant: The Pro plan costs **$49/month** (standard tier), while the Enterprise plan costs **$149/month** (premium tier). 

Comparing the two:
* **Price Difference:** The Enterprise plan is **$100 more per month** than the Pro plan.
* **Pro Plan Features:** Up to 10 users, 100 GB storage, and priority support.
* **Enterprise Plan Features:** Unlimited users, SSO, and a dedicated su

c:\Users\pc 1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': "For a budget-conscious client, you should recommend the **Pro plan**. \n\nHere is why:\n* **Lower Cost:** At **$49/month**, it is $100 cheaper than the Enterprise plan ($149/month), saving the client $1,200 annually.\n* **Adequate Limits:** It includes up to 10 users and 100 GB of storage, which is usually more than enough for small teams or startups watching their expenses. \n* **Core Support:** It still includes priority support, so they won't be left without help if issues arise. \n\nThe Enterprise plan should only be considered if the client strictly needs unlimited users, Single Sign-On (SSO), or a dedicated success manager, which aren't necessary for most budget-focused operations.", 'index': 0, 'extras': {'signature': 'El4KXAERTTIPkaun4tBCqUC+QaMTK82UBjjShdVuykUydf0yu3boFxXz+eSEna2rKxn1xtyY21uXM6YLsiR6AaIx9pTYOPiA35N02cLWWpCsyLr7Bz3CNmF2OexxbAvz'}}]

> Finished chain.

Assistant: For a budget-conscious client, you should recommend the **Pro plan**. 

H

In [22]:
from agent_setup import structured_recommendation, ProductRecommendation

draft = message_text(last["output"])

structured = structured_recommendation(
    turns[-1],
    draft
)

print(structured.model_dump_json(indent=2))

c:\Users\pc 1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{
  "recommended_product": "Pro plan",
  "price_usd": 49.0,
  "rationale": "It is $100 cheaper than the Enterprise plan, saving the client $1,200 annually while including adequate limits and priority support.",
  "budget_friendly": true
}


## Task 5 — Structured output & error handling

1. Parse the final recommendation with `with_structured_output(ProductRecommendation)`.
2. `broken_lookup` raises `ToolException`; `handle_tool_error=True` feeds the error back as an observation so the agent can recover.


In [23]:
from tools import broken_lookup
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from agent_setup import build_prompt, message_text
from config import build_llm

# Build an agent with the intentionally failing tool
llm = build_llm()

agent = create_tool_calling_agent(
    llm,
    [broken_lookup],
    build_prompt()
)

err_executor = AgentExecutor(
    agent=agent,
    tools=[broken_lookup],
    verbose=True,
    max_iterations=3
)

print("\n--- broken_lookup recovery ---")

err_out = err_executor.invoke({
    "input": "Call broken_lookup with query 'pricing-api' and explain what happened.",
    "chat_history": [],
})

print("\nFINAL:", message_text(err_out["output"]))


--- broken_lookup recovery ---


> Entering new AgentExecutor chain...


c:\Users\pc 1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `broken_lookup` with `{'query': 'pricing-api'}`


upstream timeout while looking up 'pricing-api'

c:\Users\pc 1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'I called the `broken_lookup` tool with the query `\'pricing-api\'`, and it returned an "upstream timeout while looking up \'pricing-api\'" error message. As instructed, I am explaining this failure rather than inventing numbers.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIP+LGpB05qMiP7aWyG6xvGyI3b0m+KNaU7v6MH9+pdfHrHmF3qkhU+vTnJKqZIjf3M/D4KbgVOUcu9kY9jP+7AKJgbN8v87L/h0PsJwM5kpQc/u58RH6GY'}}]

> Finished chain.

FINAL: I called the `broken_lookup` tool with the query `'pricing-api'`, and it returned an "upstream timeout while looking up 'pricing-api'" error message. As instructed, I am explaining this failure rather than inventing numbers.


In [24]:
reflection = '''
LangChain made tool registration, the agent loop, memory, and structured output much easier than Day 1's raw Python agent.
It provided reusable abstractions for tools, agents, memory, and structured responses instead of requiring me to manually manage each part.
The main abstraction leakiness I noticed was dealing with LangChain's import paths, Gemini content blocks, and model-specific requirements such as thought signatures.
Verbose mode exposes the agent/tool interaction, but some of the internal reasoning and orchestration is still hidden compared with my raw-Python implementation.
'''

print(reflection)


LangChain made tool registration, the agent loop, memory, and structured output much easier than Day 1's raw Python agent.
It provided reusable abstractions for tools, agents, memory, and structured responses instead of requiring me to manually manage each part.
The main abstraction leakiness I noticed was dealing with LangChain's import paths, Gemini content blocks, and model-specific requirements such as thought signatures.
Verbose mode exposes the agent/tool interaction, but some of the internal reasoning and orchestration is still hidden compared with my raw-Python implementation.

